In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [2]:
X,y = make_classification(n_samples=1000,
                          n_features=10,
                          n_redundant=8,
                          weights=[0.9,0.1],
                          flip_y=0,
                          random_state=42)
np.unique(y,return_counts=True)

(array([0, 1]), array([900, 100]))

In [3]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,stratify=y,random_state=42)

#### Experiment 1: Train Logistic Regression

In [4]:
params={
    'solver':'lbfgs',
    'max_iter':1000,
    'multi_class':'auto',
    'random_state':8888
}
lr=LogisticRegression(**params)
lr.fit(X_train,y_train)
y_pred=lr.predict(X_test)
report=classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.95      0.97      0.96       270
           1       0.62      0.50      0.56        30

    accuracy                           0.92       300
   macro avg       0.79      0.73      0.76       300
weighted avg       0.91      0.92      0.92       300



In [5]:
report_dict=classification_report(y_test,y_pred,output_dict=True)
report_dict

{'0': {'precision': 0.9456521739130435,
  'recall': 0.9666666666666667,
  'f1-score': 0.9560439560439561,
  'support': 270.0},
 '1': {'precision': 0.625,
  'recall': 0.5,
  'f1-score': 0.5555555555555556,
  'support': 30.0},
 'accuracy': 0.92,
 'macro avg': {'precision': 0.7853260869565217,
  'recall': 0.7333333333333334,
  'f1-score': 0.7557997557997558,
  'support': 300.0},
 'weighted avg': {'precision': 0.9135869565217392,
  'recall': 0.92,
  'f1-score': 0.9159951159951161,
  'support': 300.0}}

In [6]:
import mlflow

#### Experiment 2: Train Random Forest Classifier

In [7]:
rf_clf= RandomForestClassifier(n_estimators=50)
rf_clf.fit(X_train,y_train)
y_pred=rf_clf.predict(X_test)
report=classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.83      0.89        30

    accuracy                           0.98       300
   macro avg       0.97      0.91      0.94       300
weighted avg       0.98      0.98      0.98       300



#### Experiment 3: Train XGBoost

In [8]:
xgb_clf=XGBClassifier(use_label_encoder=False,eval_metrics='logloss')
xgb_clf.fit(X_train,y_train)
y_pred_xgb=xgb_clf.predict(X_test)
print(classification_report(y_test,y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



#### Experiment 4: Handle class Imbalance using SMOTETomek and then train XGBoost

In [9]:
from imblearn.combine import SMOTETomek
import mlflow.sklearn
import mlflow.xgboost

sat= SMOTETomek(random_state=42)
X_train_res,y_train_res = sat.fit_resample(X_train,y_train)
np.unique(y_train_res,return_counts=True)

(array([0, 1]), array([619, 619]))

In [10]:
#dagshub sethub
import dagshub
dagshub.init(repo_owner='akinluaayomide8', repo_name='mlflow_dagshub_demo', mlflow=True)

import mlflow
#with mlflow.start_run():
  #mlflow.log_param('parameter name', 'value')
  #mlflow.log_metric('metric name', 1)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=1726d543-77fe-4998-b428-8a42d7afdd8a&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=a6604074a9d413d7e10dd1c3b8250436622c7d442fea13526daa2eb54fc6d2a0




Output()

Accessing as akinluaayomide8

Initialized MLflow to track repo "akinluaayomide8/mlflow_dagshub_demo"

Repository akinluaayomide8/mlflow_dagshub_demo initialized!

In [11]:
xgb_clf.fit(X_train_res,y_train_res)
y_pred_xgb_smote=xgb_clf.predict(X_test)
print(classification_report(y_test,y_pred_xgb_smote))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



In [12]:
models=[
    (
      'LogisticRegression',
        LogisticRegression(C=1,solver='liblinear'),
        (X_train,y_train),
        (X_test,y_test)
    ),
    (
        'Random Forest',
        RandomForestClassifier(n_estimators=30,max_depth=3),
        (X_train,y_train),
        (X_test,y_test)
    ),
    (
        'XGBClassifier',
        XGBClassifier(use_label_encoder=False,eval_metrics='logloss'),
        (X_train,y_train),
        (X_test,y_test)
    ),
    (
        'XGBClassifier with SMOTE',
        XGBClassifier(use_label_encoder=False,eval_metrics='logloss'),
        (X_train_res,y_train_res),
        (X_test,y_test)
    )
]

In [13]:
reports= []
for model_name,model,train_set,test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]

    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test,y_pred,output_dict=True)
    reports.append(report)

In [14]:
reports

[{'0': {'precision': 0.9454545454545454,
   'recall': 0.9629629629629629,
   'f1-score': 0.9541284403669725,
   'support': 270.0},
  '1': {'precision': 0.6,
   'recall': 0.5,
   'f1-score': 0.5454545454545454,
   'support': 30.0},
  'accuracy': 0.9166666666666666,
  'macro avg': {'precision': 0.7727272727272727,
   'recall': 0.7314814814814814,
   'f1-score': 0.749791492910759,
   'support': 300.0},
  'weighted avg': {'precision': 0.9109090909090909,
   'recall': 0.9166666666666666,
   'f1-score': 0.91326105087573,
   'support': 300.0}},
 {'0': {'precision': 0.9607142857142857,
   'recall': 0.9962962962962963,
   'f1-score': 0.9781818181818182,
   'support': 270.0},
  '1': {'precision': 0.95,
   'recall': 0.6333333333333333,
   'f1-score': 0.76,
   'support': 30.0},
  'accuracy': 0.96,
  'macro avg': {'precision': 0.9553571428571428,
   'recall': 0.8148148148148149,
   'f1-score': 0.8690909090909091,
   'support': 300.0},
  'weighted avg': {'precision': 0.9596428571428572,
   'recall':

In [15]:
print(f'is {mlflow.get_tracking_uri()}')

is https://dagshub.com/akinluaayomide8/mlflow_dagshub_demo.mlflow


In [16]:
import dagshub.auth
print(dagshub.auth.get_token())

cc02875b26a560898514e0478d9341d1f798378b


In [19]:
import os
os.environ['MLFLOW_TRACKING_USERNAME']='akinluaayomide8'
os.environ['MLFLOW_TRACKING_PASSWORD']='cc02875b26a560898514e0478d9341d1f798378b'
os.environ['MLFLOW_TRACKING_URI']='https://dagshub.com/akinluaayomide8/mlflow_dagshub_demo.mlflow'

In [22]:
import joblib
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment('Anomaly_detection')
for i,element in enumerate(models):
    model_name=element[0]
    model=element[1]
    report=reports[i]
    with mlflow.start_run(run_name=model_name):
        mlflow.log_param('model_name',model_name)
        mlflow.log_metrics({
        'accuracy':report['accuracy'],
        'recall_class_0':report['0']['recall'],
        'recall_class_1':report['1']['recall'],
        'f1_score_class_0':report['0']['f1-score'],
        'f1_score_class_1':report['1']['f1-score'],
        'f1_score_macro_avg':report['macro avg']['f1-score']
         })
        joblib.dump(model,f'{model_name}.pkl')
        mlflow.log_artifact(f'{model_name}.pkl',artifact_path='models')
        #if 'XGB' in model_name:
             #mlflow.xgboost.log_model(model,f'{model_name}')
        #else:
             #mlflow.sklearn.log_(model,f'{model_name}')
        

🏃 View run LogisticRegression at: https://dagshub.com/akinluaayomide8/mlflow_dagshub_demo.mlflow/#/experiments/0/runs/9226933494494b2f85e73d07305698fb
🧪 View experiment at: https://dagshub.com/akinluaayomide8/mlflow_dagshub_demo.mlflow/#/experiments/0
🏃 View run Random Forest at: https://dagshub.com/akinluaayomide8/mlflow_dagshub_demo.mlflow/#/experiments/0/runs/279877f6d91d4403b1eb709a11eabe39
🧪 View experiment at: https://dagshub.com/akinluaayomide8/mlflow_dagshub_demo.mlflow/#/experiments/0
🏃 View run XGBClassifier at: https://dagshub.com/akinluaayomide8/mlflow_dagshub_demo.mlflow/#/experiments/0/runs/56c751205594468aa3562353a9a73e6a
🧪 View experiment at: https://dagshub.com/akinluaayomide8/mlflow_dagshub_demo.mlflow/#/experiments/0
🏃 View run XGBClassifier with SMOTE at: https://dagshub.com/akinluaayomide8/mlflow_dagshub_demo.mlflow/#/experiments/0/runs/feb9fc3806ca4fb3bf306526ddf95be9
🧪 View experiment at: https://dagshub.com/akinluaayomide8/mlflow_dagshub_demo.mlflow/#/experiment

In [39]:
exp = mlflow.get_experiment_by_name("Anomaly_detection")
print(exp)

None


#### Register the model

In [23]:
model_name = 'LogisticRegression'
run_id = input('Enter Run ID')
model_uri=f'runs:/{run_id}/{model_name}'
result = mlflow.register_model(
     model_uri,model_name
)

Enter Run ID 56c751205594468aa3562353a9a73e6a


Successfully registered model 'LogisticRegression'.


RestException: INTERNAL_ERROR: Response: {'error': 'unsupported endpoint, please contact support@dagshub.com'}

#### Load the model

In [33]:
model_version = 1
model_uri = f'models:/{model_name}@challenger'
loaded_model = mlflow.sklearn.load_model(model_uri)
y_pred=loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [34]:
result

<ModelVersion: aliases=[], creation_timestamp=1764262351846, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1764262351846, metrics=None, model_id=None, name='LogisticRegression', params=None, run_id='76056649f7054a3aa0ee7d4fbe4b3e85', run_link='', source='models:/m-a70a8d8a2a5b4cb284a943d912044dbb', status='READY', status_message=None, tags={}, user_id='', version='1'>

In [36]:
dev_model_uri = f'models:/{model_name}@challenger'
prod_model='anomaly-detection-prod'
client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=dev_model_uri,dst_name=prod_model)

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'LogisticRegression' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1764265533830, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1764265533830, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='76056649f7054a3aa0ee7d4fbe4b3e85', run_link='', source='models:/LogisticRegression/1', status='READY', status_message=None, tags={}, user_id='', version='1'>